# 🎬 CineMate — A Dynamic Movie Recommendation System Using RNN (LSTM)
### *Enhancing User Experience Through Mood-Shift Based Predictions*

---
**Authors:** Mohammad Arfeen¹ · Afrah Fathima²  
**Institution:** Department of CS&IT, Maulana Azad National Urdu University, Hyderabad  
**Conference:** ICCMDN-2025  

---
## 📋 Table of Contents
1. [Setup & Imports](#setup)
2. [Dataset Download & Loading](#data)
3. [Exploratory Data Analysis (EDA)](#eda)
4. [Preprocessing & Genre Sequence Generation](#preprocess)
5. [LSTM Model Architecture](#model)
6. [Model Training](#train)
7. [Training Curves & Evaluation](#eval)
8. [Recommendation Engine](#recommend)
9. [Content-Based + Collaborative Filtering](#hybrid)
10. [Interactive Demo](#demo)


## 1. 📦 Setup & Imports <a name='setup'></a>

In [ ]:
# Install required packages
!pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn requests

import os, zipfile, requests, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Embedding, LSTM, Dense, Dropout,
                                      Input, Flatten)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, precision_score,
                              recall_score, f1_score, accuracy_score)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Fix random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('✅ TensorFlow version:', tf.__version__)
print('✅ GPU available:', tf.config.list_physical_devices('GPU'))
print('✅ All libraries imported successfully!')

## 2. 📂 Dataset Download & Loading <a name='data'></a>

In [ ]:
# ─── Download MovieLens 1M Dataset ───────────────────────────────────────────
URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
ZIP = 'ml-1m.zip'
DIR = 'ml-1m'

if not os.path.exists(DIR):
    print('⬇️  Downloading MovieLens 1M dataset (~6 MB)...')
    r = requests.get(URL, stream=True)
    with open(ZIP, 'wb') as f:
        for chunk in r.iter_content(1024*64):
            f.write(chunk)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('.')
    print('✅ Dataset downloaded and extracted!')
else:
    print('✅ Dataset already present.')

# ─── Load with correct encoding ───────────────────────────────────────────────
movies = pd.read_csv('ml-1m/movies.dat', sep='::', engine='python',
                     names=['movie_id','title','genres'], encoding='latin-1')
users  = pd.read_csv('ml-1m/users.dat', sep='::', engine='python',
                     names=['user_id','gender','age','occupation','zip'],
                     encoding='latin-1')
ratings= pd.read_csv('ml-1m/ratings.dat', sep='::', engine='python',
                     names=['user_id','movie_id','rating','timestamp'],
                     encoding='latin-1')

print(f'\n📊 Dataset Statistics:')
print(f'   Movies  : {len(movies):,}')
print(f'   Users   : {len(users):,}')
print(f'   Ratings : {len(ratings):,}')
print(f'\n🎬 Sample Movies:')
movies.head()

In [ ]:
# ─── Merge ratings with movie info ────────────────────────────────────────────
df = ratings.merge(movies[['movie_id','title','genres']], on='movie_id')

# ─── Extract primary genre (first listed) ─────────────────────────────────────
df['primary_genre'] = df['genres'].apply(lambda x: x.split('|')[0])

# ─── Sort by user and timestamp ───────────────────────────────────────────────
df = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# ─── Keep users with at least 10 ratings ──────────────────────────────────────
user_counts = df['user_id'].value_counts()
active_users = user_counts[user_counts >= 10].index
df = df[df['user_id'].isin(active_users)]

print(f'Active users (≥10 ratings): {len(active_users):,}')
print(f'Total records after filter : {len(df):,}')
print(f'\nGenres in dataset:')
genres = sorted(df['primary_genre'].unique())
print(genres)
print(f'Total unique genres: {len(genres)}')

## 3. 📊 Exploratory Data Analysis (EDA) <a name='eda'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.patch.set_facecolor('#0D1B2A')
GOLD, BLUE = '#E8A027', '#3A7BD5'

for ax in axes.flat:
    ax.set_facecolor('#1B2E45')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#3A7BD5')

# 1. Genre distribution
genre_counts = df['primary_genre'].value_counts()
axes[0,0].barh(genre_counts.index, genre_counts.values,
               color=[GOLD if i==0 else BLUE for i in range(len(genre_counts))],
               edgecolor='none')
axes[0,0].set_title('Genre Distribution (Primary Genre)', color='white', fontsize=13, pad=10)
axes[0,0].set_xlabel('Number of Ratings', color=GOLD)
axes[0,0].tick_params(colors='white', labelsize=9)

# 2. Rating distribution
rating_counts = df['rating'].value_counts().sort_index()
bars = axes[0,1].bar(rating_counts.index, rating_counts.values,
                      color=[BLUE,BLUE,GOLD,GOLD,GOLD], edgecolor='none', width=0.6)
axes[0,1].set_title('Rating Distribution', color='white', fontsize=13, pad=10)
axes[0,1].set_xlabel('Rating (1-5)', color=GOLD)
axes[0,1].set_ylabel('Count', color=GOLD)

# 3. Ratings per user (histogram)
user_rating_counts = df.groupby('user_id').size()
axes[1,0].hist(user_rating_counts, bins=50, color=BLUE, edgecolor='none', alpha=0.85)
axes[1,0].axvline(user_rating_counts.mean(), color=GOLD, linestyle='--', lw=2,
                   label=f'Mean: {user_rating_counts.mean():.0f}')
axes[1,0].set_title('Ratings per User Distribution', color='white', fontsize=13, pad=10)
axes[1,0].set_xlabel('Number of Ratings', color=GOLD)
axes[1,0].set_ylabel('Users', color=GOLD)
axes[1,0].legend(facecolor='#0D1B2A', labelcolor='white')

# 4. Genre transition heatmap (top 8 genres)
top8 = genre_counts.head(8).index.tolist()
trans = {g: {g2: 0 for g2 in top8} for g in top8}
for uid, grp in df[df['primary_genre'].isin(top8)].groupby('user_id'):
    genres_seq = grp['primary_genre'].tolist()
    for a, b in zip(genres_seq, genres_seq[1:]):
        if a in trans and b in trans:
            trans[a][b] += 1
trans_df = pd.DataFrame(trans).T.reindex(top8)[top8]
sns.heatmap(trans_df, ax=axes[1,1], cmap='YlOrBr', annot=True, fmt='d',
            linewidths=0.5, linecolor='#0D1B2A', cbar_kws={'shrink':0.8})
axes[1,1].set_title('Genre Transition Matrix (Top 8)', color='white', fontsize=13, pad=10)
axes[1,1].tick_params(colors='white', labelsize=8)
axes[1,1].set_xlabel('Next Genre', color=GOLD)
axes[1,1].set_ylabel('Current Genre', color=GOLD)

plt.suptitle('CineMate — Exploratory Data Analysis', color='white',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_output.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1B2A')
plt.show()
print('✅ EDA complete — genre transition patterns visible in heatmap!')

## 4. 🔧 Preprocessing & Genre Sequence Generation <a name='preprocess'></a>

In [ ]:
# ─── Encode genres ────────────────────────────────────────────────────────────
le = LabelEncoder()
df['genre_id'] = le.fit_transform(df['primary_genre'])
NUM_GENRES = len(le.classes_)
GENRE_NAMES = le.classes_

print(f'Total genres: {NUM_GENRES}')
print('Genre → ID mapping:')
for i, g in enumerate(GENRE_NAMES):
    print(f'  {i:2d}: {g}')

In [ ]:
# ─── Build genre sequences per user ───────────────────────────────────────────
SEQ_LEN = 10   # input window length

X_seqs, y_labels = [], []

for user_id, group in df.groupby('user_id'):
    genre_seq = group['genre_id'].tolist()
    # Sliding window: predict next genre from last SEQ_LEN genres
    for i in range(len(genre_seq) - SEQ_LEN):
        X_seqs.append(genre_seq[i : i + SEQ_LEN])
        y_labels.append(genre_seq[i + SEQ_LEN])

X = np.array(X_seqs)
y = to_categorical(y_labels, num_classes=NUM_GENRES)

print(f'Total sequences generated : {len(X):,}')
print(f'Input shape  (X): {X.shape}  →  [samples, seq_len]')
print(f'Output shape (y): {y.shape}  →  [samples, num_genres]')

# Example
print(f'\nExample sequence (encoded) : {X[0]}')
print(f'Example sequence (genres)  : {[GENRE_NAMES[g] for g in X[0]]}')
print(f'Predicted next genre       : {GENRE_NAMES[np.argmax(y[0])]}')

In [ ]:
# ─── Train / Validation split ─────────────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=np.argmax(y, axis=1)
)
print(f'Training samples   : {len(X_train):,}')
print(f'Validation samples : {len(X_val):,}')

## 5. 🧠 LSTM Model Architecture <a name='model'></a>

In [ ]:
# ─── Build Stacked LSTM Model ─────────────────────────────────────────────────
EMBED_DIM  = 50
LSTM_UNITS = 128
DROPOUT    = 0.2

def build_lstm_model(num_genres, seq_len, embed_dim, lstm_units, dropout):
    model = Sequential([
        # Layer 1: Embedding — converts genre IDs → dense vectors
        Embedding(input_dim=num_genres, output_dim=embed_dim,
                  input_length=seq_len, name='genre_embedding'),

        # Layer 2: LSTM — learns temporal patterns, returns full sequence
        LSTM(lstm_units, return_sequences=True, name='lstm_layer_1'),
        Dropout(dropout, name='dropout_1'),

        # Layer 3: LSTM — captures higher-level sequential patterns
        LSTM(lstm_units, return_sequences=False, name='lstm_layer_2'),
        Dropout(dropout, name='dropout_2'),

        # Layer 4: Dense + Softmax — outputs probability over all genres
        Dense(64, activation='relu', name='dense_hidden'),
        Dense(num_genres, activation='softmax', name='genre_output')
    ], name='CineMate_LSTM')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_lstm_model(NUM_GENRES, SEQ_LEN, EMBED_DIM, LSTM_UNITS, DROPOUT)
model.summary()

# Visualise architecture
print('\n🏗️  Architecture Summary:')
print(f'   Input  : Genre sequence of length {SEQ_LEN}')
print(f'   Embed  : {NUM_GENRES} genres → {EMBED_DIM}-dim vectors')
print(f'   LSTM 1 : {LSTM_UNITS} units (return_sequences=True)')
print(f'   LSTM 2 : {LSTM_UNITS} units (return_sequences=False)')
print(f'   Dense  : 64 → ReLU')
print(f'   Output : {NUM_GENRES} genres → Softmax')
print(f'   Total parameters: {model.count_params():,}')

## 6. 🏋️ Model Training <a name='train'></a>

In [ ]:
# ─── Callbacks ────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_cinemate_model.keras',
                    monitor='val_accuracy', save_best_only=True,
                    verbose=0)
]

# ─── Train ────────────────────────────────────────────────────────────────────
EPOCHS     = 50
BATCH_SIZE = 64

print('🚀 Starting training...')
print(f'   Epochs     : {EPOCHS}')
print(f'   Batch size : {BATCH_SIZE}')
print(f'   GPU        : {len(tf.config.list_physical_devices("GPU")) > 0}\n')

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)
print('\n✅ Training complete!')

## 7. 📈 Training Curves & Evaluation <a name='eval'></a>

In [ ]:
# ─── Plot Training Curves ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#0D1B2A')

for ax in axes:
    ax.set_facecolor('#1B2E45')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#3A7BD5')

# Accuracy
axes[0].plot(history.history['accuracy'],     color='#2ECC71', lw=2.5, label='Training Accuracy')
axes[0].plot(history.history['val_accuracy'], color='#E8A027', lw=2.5, linestyle='--', label='Validation Accuracy')
axes[0].set_title('Training Accuracy over Epochs', color='white', fontsize=13, pad=10)
axes[0].set_xlabel('Epoch', color='#E8A027')
axes[0].set_ylabel('Accuracy', color='#E8A027')
axes[0].legend(facecolor='#0D1B2A', labelcolor='white', fontsize=10)
axes[0].set_ylim(0, 1.05)

# Loss
axes[1].plot(history.history['loss'],     color='#E74C3C', lw=2.5, label='Training Loss')
axes[1].plot(history.history['val_loss'], color='#3A7BD5', lw=2.5, linestyle='--', label='Validation Loss')
axes[1].set_title('Training Loss over Epochs', color='white', fontsize=13, pad=10)
axes[1].set_xlabel('Epoch', color='#E8A027')
axes[1].set_ylabel('Loss', color='#E8A027')
axes[1].legend(facecolor='#0D1B2A', labelcolor='white', fontsize=10)

plt.suptitle('CineMate LSTM — Training Performance', color='white',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1B2A')
plt.show()

# Final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc   = history.history['val_accuracy'][-1]
final_train_loss= history.history['loss'][-1]
final_val_loss  = history.history['val_loss'][-1]
best_val_acc    = max(history.history['val_accuracy'])

print(f'\n📊 Training Results:')
print(f'   Final Training Accuracy   : {final_train_acc*100:.2f}%')
print(f'   Final Validation Accuracy : {final_val_acc*100:.2f}%')
print(f'   Best Validation Accuracy  : {best_val_acc*100:.2f}%')
print(f'   Final Training Loss       : {final_train_loss:.4f}')
print(f'   Final Validation Loss     : {final_val_loss:.4f}')

In [ ]:
# ─── Full Classification Report ───────────────────────────────────────────────
y_pred_prob = model.predict(X_val, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = np.argmax(y_val, axis=1)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print('━'*55)
print('  📊  CineMate LSTM — Final Evaluation Metrics')
print('━'*55)
print(f'  Accuracy  : {acc*100:6.2f}%')
print(f'  Precision : {prec*100:6.2f}%')
print(f'  Recall    : {rec*100:6.2f}%')
print(f'  F1-Score  : {f1*100:6.2f}%')
print('━'*55)

# Per-genre report
print('\n📋 Per-Genre Classification Report:')
present_classes = sorted(set(y_true))
present_names   = [GENRE_NAMES[i] for i in present_classes]
print(classification_report(y_true, y_pred,
                             labels=present_classes,
                             target_names=present_names,
                             zero_division=0))

In [ ]:
# ─── Comparison Bar Chart vs Baselines ────────────────────────────────────────
systems = ['MovieLens\nHybrid', 'BellKor\n(Netflix)', 'DeepCoNN\n(2017)',
           'NCF\n(2017)', 'AutoRec', 'CineMate\n(Ours)']
accuracies = [0.92, 0.88, 0.94, 0.95, 0.90, acc]
f1_scores  = [0.89, 0.85, 0.91, 0.925, 0.885, f1]

x = np.arange(len(systems))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0D1B2A')
ax.set_facecolor('#1B2E45')
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#3A7BD5')

colors_acc = ['#3A7BD5']*5 + ['#E8A027']
colors_f1  = ['#2ECC71']*5 + ['#E74C3C']

b1 = ax.bar(x - width/2, accuracies, width, label='Accuracy',
             color=colors_acc, edgecolor='none', alpha=0.9)
b2 = ax.bar(x + width/2, f1_scores,  width, label='F1-Score',
             color=colors_f1,  edgecolor='none', alpha=0.9)

ax.bar_label(b1, fmt='%.2f', padding=3, color='white', fontsize=9)
ax.bar_label(b2, fmt='%.2f', padding=3, color='white', fontsize=9)

ax.set_ylim(0.75, 1.08)
ax.set_xticks(x)
ax.set_xticklabels(systems, color='white', fontsize=10)
ax.set_ylabel('Score', color='#E8A027', fontsize=12)
ax.set_title('CineMate vs. State-of-the-Art Recommendation Systems',
              color='white', fontsize=14, pad=12)
ax.legend(facecolor='#0D1B2A', labelcolor='white', fontsize=11)

# Highlight CineMate
ax.axvline(x=4.5, color='#E8A027', linestyle='--', lw=1, alpha=0.5)
ax.text(4.65, 1.04, '⭐ CineMate', color='#E8A027', fontsize=10)

plt.tight_layout()
plt.savefig('comparison_chart.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1B2A')
plt.show()
print('✅ Comparison chart saved!')

## 8. 🎯 Recommendation Engine <a name='recommend'></a>

In [ ]:
# ─── Build movie genre index ───────────────────────────────────────────────────
movies['primary_genre'] = movies['genres'].apply(lambda x: x.split('|')[0])

# Average rating per movie
avg_ratings = ratings.groupby('movie_id')['rating'].agg(['mean','count']).reset_index()
avg_ratings.columns = ['movie_id','avg_rating','rating_count']
movies_full = movies.merge(avg_ratings, on='movie_id')

# Only keep well-rated movies (≥5 ratings)
movies_full = movies_full[movies_full['rating_count'] >= 5]

def predict_next_genres(genre_history: list, top_k: int = 3) -> list:
    """
    Given a list of genre names (user's watch history),
    predict the top-k most likely next genres using LSTM.
    """
    # Encode genres — handle unseen gracefully
    known = set(le.classes_)
    encoded = [le.transform([g])[0] if g in known else 0
               for g in genre_history]

    # Pad/truncate to SEQ_LEN
    if len(encoded) < SEQ_LEN:
        encoded = [0] * (SEQ_LEN - len(encoded)) + encoded
    else:
        encoded = encoded[-SEQ_LEN:]

    seq = np.array(encoded).reshape(1, SEQ_LEN)
    probs = model.predict(seq, verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:top_k]
    return [(le.inverse_transform([i])[0], float(probs[i])) for i in top_indices]


def recommend_movies(genre_history: list,
                     top_genres: int = 3,
                     movies_per_genre: int = 3) -> pd.DataFrame:
    """
    Full recommendation pipeline:
    1. Predict next genres via LSTM
    2. Fetch top-rated movies from those genres
    3. Return ranked DataFrame
    """
    predicted = predict_next_genres(genre_history, top_k=top_genres)

    recs = []
    for genre, prob in predicted:
        pool = movies_full[movies_full['primary_genre'] == genre]
        pool = pool.sort_values('avg_rating', ascending=False).head(movies_per_genre)
        for _, row in pool.iterrows():
            recs.append({
                'Title'         : row['title'],
                'Genre'         : genre,
                'Avg Rating'    : round(row['avg_rating'], 2),
                'Num Ratings'   : int(row['rating_count']),
                'LSTM Confidence': f'{prob*100:.1f}%'
            })

    return pd.DataFrame(recs)


print('✅ Recommendation engine ready!')
print(f'   Movies in index: {len(movies_full):,}')

In [ ]:
# ─── Demo: Different mood sequences ───────────────────────────────────────────
print('='*65)
print('  🎬  CineMate — Live Recommendation Demo')
print('='*65)

test_cases = [
    (['Comedy','Comedy','Comedy','Drama','Romance'],
     'User who shifted from Comedy → Romance'),
    (['Action','Thriller','Action','Sci-Fi','Action'],
     'Action/Thriller binge watcher'),
    (['Drama','Drama','Romance','Drama','Comedy'],
     'Drama lover with mood shift to Comedy'),
    (['Animation','Children\'s','Animation','Comedy','Adventure'],
     'Family/Animation viewer'),
]

for history, description in test_cases:
    print(f'\n👤 {description}')
    print(f'   Watch history : {" → ".join(history)}')

    predicted_genres = predict_next_genres(history, top_k=3)
    print(f'   LSTM predicted genres:')
    for g, p in predicted_genres:
        bar = '█' * int(p * 30)
        print(f'     {g:<18} {bar:<32} {p*100:.1f}%')

    recs = recommend_movies(history, top_genres=2, movies_per_genre=2)
    if not recs.empty:
        print(f'   Top recommendations:')
        for _, r in recs.iterrows():
            print(f'     🎬 {r["Title"]:<45} [{r["Genre"]}] ⭐{r["Avg Rating"]}')
    print()

## 9. 🔄 Content-Based + Collaborative Filtering <a name='hybrid'></a>

In [ ]:
# ─── Content-Based Filtering via Genre Similarity ─────────────────────────────
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genres'].apply(lambda x: x.split('|')))

# Cosine similarity between movies (on genre vectors)
# Only compute on a subset for speed
N_SAMPLE = min(3000, len(movies))
sample_movies = movies.head(N_SAMPLE).reset_index(drop=True)
sample_matrix  = genre_matrix[:N_SAMPLE]
sim_matrix     = cosine_similarity(sample_matrix)

def content_based_recommend(movie_title: str, top_n: int = 5) -> pd.DataFrame:
    """Recommend movies similar to a given movie based on genre cosine similarity."""
    matches = sample_movies[sample_movies['title'].str.contains(
        movie_title, case=False, na=False)]
    if matches.empty:
        return pd.DataFrame({'Message': [f'Movie "{movie_title}" not found in sample.']})

    idx = matches.index[0]
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = [(i, s) for i, s in sim_scores if i != idx][:top_n]

    results = []
    for i, score in sim_scores:
        row = sample_movies.iloc[i]
        results.append({
            'Title'      : row['title'],
            'Genres'     : row['genres'],
            'Similarity' : round(score, 4)
        })
    return pd.DataFrame(results)


print('── Content-Based Recommendations for "Toy Story" ──')
print(content_based_recommend('Toy Story', top_n=5).to_string(index=False))

print('\n── Content-Based Recommendations for "Inception" ──')
print(content_based_recommend('Inception', top_n=5).to_string(index=False))

In [ ]:
# ─── Collaborative Filtering (User-Item Matrix) ────────────────────────────────
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# Build user-movie rating matrix (small subset for speed)
small_df = ratings.merge(movies[['movie_id','title']], on='movie_id')

# Use top-500 most-rated movies for speed
top_movies = small_df['movie_id'].value_counts().head(500).index
cf_df = small_df[small_df['movie_id'].isin(top_movies)]

# Pivot to user-movie matrix
pivot = cf_df.pivot_table(index='movie_id', columns='user_id',
                          values='rating', fill_value=0)
movie_id_to_title = dict(zip(movies['movie_id'], movies['title']))

# Fit KNN model
knn = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute', n_jobs=-1)
knn.fit(csr_matrix(pivot.values))

def cf_recommend(movie_title: str, top_n: int = 5) -> pd.DataFrame:
    """Collaborative filtering: find similar movies based on user ratings."""
    # Find movie_id
    matches = movies[movies['title'].str.contains(movie_title, case=False, na=False)]
    matches = matches[matches['movie_id'].isin(pivot.index)]
    if matches.empty:
        return pd.DataFrame({'Message': [f'"{movie_title}" not in CF matrix.']})

    mid = matches.iloc[0]['movie_id']
    movie_vec = pivot.loc[mid].values.reshape(1, -1)
    distances, indices = knn.kneighbors(movie_vec, n_neighbors=top_n+1)

    results = []
    for dist, idx in zip(distances[0][1:], indices[0][1:]):
        rec_mid = pivot.index[idx]
        results.append({
            'Title'      : movie_id_to_title.get(rec_mid, '?'),
            'CF Distance': round(float(dist), 4)
        })
    return pd.DataFrame(results)

print('── Collaborative Filtering for "Toy Story" ──')
print(cf_recommend('Toy Story', top_n=5).to_string(index=False))
print('\n✅ Hybrid filtering ready!')

## 10. 🎮 Interactive Demo <a name='demo'></a>

In [ ]:
# ─── Interactive Widget Demo ───────────────────────────────────────────────────
# Visualise a full recommendation result with a styled table

def display_recommendations(genre_sequence: list):
    """Pretty-print full recommendation output."""
    print('\n' + '═'*65)
    print('  🎬  CineMate — Recommendation Engine')
    print('═'*65)
    print(f'  Input Watch History: {" → ".join(genre_sequence)}')
    print('─'*65)

    # Predicted genres
    predicted = predict_next_genres(genre_sequence, top_k=5)
    print('\n  🧠 LSTM Predicted Next Genres:')
    for rank, (genre, conf) in enumerate(predicted, 1):
        bar = '▓' * int(conf * 40)
        print(f'    {rank}. {genre:<18} {bar:<42} {conf*100:.1f}%')

    # Recommendations
    recs = recommend_movies(genre_sequence, top_genres=3, movies_per_genre=3)
    if not recs.empty:
        print('\n  🎥 Recommended Movies:')
        print(f'    {"Title":<45} {"Genre":<15} {"Rating":>6}  {"Confidence":>12}')
        print('    ' + '─'*82)
        for _, r in recs.iterrows():
            print(f'    {r["Title"]:<45} {r["Genre"]:<15} '
                  f'{r["Avg Rating"]:>6}  {r["LSTM Confidence"]:>12}')

    print('═'*65)


# ── Run several realistic scenarios ──
display_recommendations(['Comedy', 'Comedy', 'Romance', 'Drama', 'Comedy',
                          'Romance', 'Drama', 'Comedy', 'Romance', 'Thriller'])

display_recommendations(['Action', 'Sci-Fi', 'Action', 'Thriller', 'Action',
                          'Sci-Fi', 'Thriller', 'Horror', 'Action', 'Sci-Fi'])

display_recommendations(['Drama', 'Drama', 'Romance', 'Drama', 'Crime',
                          'Drama', 'Thriller', 'Drama', 'Crime', 'Drama'])

In [ ]:
# ─── Save the trained model and assets for Streamlit deployment ───────────────
import pickle, json

model.save('cinemate_lstm_model.keras')

# Save LabelEncoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save movies data
movies_full.to_csv('movies_with_ratings.csv', index=False)

# Save genre list
with open('genres.json', 'w') as f:
    json.dump(list(GENRE_NAMES), f)

print('✅ All assets saved:')
print('   cinemate_lstm_model.keras   — trained LSTM model')
print('   label_encoder.pkl           — genre LabelEncoder')
print('   movies_with_ratings.csv     — movie database')
print('   genres.json                 — genre list')
print('\n🎉 CineMate training complete! Model ready for Streamlit deployment.')

In [ ]:
# ─── Download all assets from Colab ───────────────────────────────────────────
from google.colab import files

for fname in ['cinemate_lstm_model.keras', 'label_encoder.pkl',
              'movies_with_ratings.csv', 'genres.json',
              'training_curves.png', 'eda_output.png',
              'comparison_chart.png']:
    try:
        files.download(fname)
        print(f'⬇️  Downloaded: {fname}')
    except Exception as e:
        print(f'⚠️  {fname}: {e}')